In [1]:
from gitsource import GithubRepositoryDataReader

reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)

files = reader.read()

documents = []

for file in files:
    doc = file.parse()
    documents.append(doc)

len(documents)

72

In [2]:
import minsearch

index = minsearch.Index(
    text_fields=["content"],
    keyword_fields=["filename"]
)

index.fit(documents)

query = "How does the agentic loop keep calling the model until it stops?"

results = index.search(query, num_results=5)

results[0]["filename"]

'01-agentic-rag/lessons/14-agentic-loop.md'

In [3]:
!wget https://raw.githubusercontent.com/DataTalksClub/llm-zoomcamp/main/01-agentic-rag/code/rag_helper.py

--2026-06-08 17:13:18--  https://raw.githubusercontent.com/DataTalksClub/llm-zoomcamp/main/01-agentic-rag/code/rag_helper.py
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 2134 (2.1K) [text/plain]
Saving to: ‘rag_helper.py.1’

rag_helper.py.1     100%[===================>]   2.08K  --.-KB/s    in 0s      

2026-06-08 17:13:18 (30.6 MB/s) - ‘rag_helper.py.1’ saved [2134/2134]



In [6]:
from importlib import reload
import rag_helper

reload(rag_helper)

<module 'rag_helper' from '/workspaces/llm-zoomcamp-2026/llm-zoomcamp-code/rag_helper.py'>

In [10]:
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()
openai_client = OpenAI()

In [11]:
import inspect
from rag_helper import RAGBase

print(inspect.signature(RAGBase))

(index, llm_client, instructions='\nYour task is to answer questions from the course participants\nbased on the provided context.\n\nUse the context to find relevant information and provide accurate\nanswers. If the answer is not found in the context,\nrespond with "I don\'t know."\n', prompt_template='QUESTION: {question}\n\nCONTEXT:\n{context}', course='llm-zoomcamp', model='gpt-5.4-mini')


In [15]:
from importlib import reload
import rag_helper
reload(rag_helper)

from rag_helper import RAGBase

rag = RAGBase(
    index=index,
    llm_client=openai_client,
    model="gpt-5.4-mini"
)

In [16]:
answer, tokens = rag.rag(
    "How does the agentic loop keep calling the model until it stops?"
)

print(tokens)
print(answer[:300])

7136
It keeps calling the model in a `while True` loop. After each response, it checks whether the model returned any `function_call` items:

- if yes, it runs the tool, appends the tool output to `messages`, and loops again
- if no, it breaks

So the stop condition is simply: **no function calls this tu


In [17]:
from gitsource import chunk_documents

chunks = chunk_documents(
    documents,
    size=2000,
    step=1000
)

len(chunks)

295

In [18]:
import minsearch

chunk_index = minsearch.Index(
    text_fields=["content"],
    keyword_fields=["filename"]
)

chunk_index.fit(chunks)

In [19]:
chunk_rag = RAGBase(
    index=chunk_index,
    llm_client=openai_client,
    model="gpt-5.4-mini"
)

In [20]:
answer, chunk_tokens = chunk_rag.rag(
    "How does the agentic loop keep calling the model until it stops?"
)

print(chunk_tokens)

2319


In [27]:
search_calls = 0

def search(query: str) -> list[dict]:
    """
    Search the lesson chunks.
    """
    global search_calls
    search_calls += 1

    return chunk_index.search(
        query,
        num_results=5,
        boost_dict={"content": 1.0}
    )

In [28]:
from toyaikit.tools import Tools

agent_tools = Tools()
agent_tools.add_tool(search)

In [29]:
instructions = """
You're a course teaching assistant.
Answer the student's question using the search tool.
Make multiple searches with different keywords before answering.
"""

In [24]:
instructions = """
You're a course teaching assistant. Answer the student's question using the search tool.
Make multiple searches with different keywords before answering.
"""

In [30]:
from toyaikit.llm import OpenAIClient
from toyaikit.chat import IPythonChatInterface
from toyaikit.chat.runners import OpenAIResponsesRunner

chat_interface = IPythonChatInterface()

runner = OpenAIResponsesRunner(
    tools=agent_tools,
    developer_prompt=instructions,
    chat_interface=chat_interface,
    llm_client=OpenAIClient(model="gpt-5.4-mini")
)

In [31]:
search_calls = 0

result = runner.loop(
    prompt="How does the agentic loop work, and how is it different from plain RAG?"
)

print("search calls:", search_calls)

search calls: 3
